# 2. Leitura e padronização dos arquivos das bases (A2)

Junta os CSVs exportados da Scopus e da IEEE Xplore em uma tabela única e padronizada.

**Antes de rodar:** depois da célula de preparação, abra o Google Drive e coloque os arquivos em
`curso_proknowc/entrada/scopus/` e `curso_proknowc/entrada/ieee/`. Pode haver vários arquivos por base.
Não precisa renomear: o que importa é a pasta.

In [ ]:
#@title Preparação (rode esta célula primeiro) { display-mode: "form" }
REPOSITORIO = "https://github.com/Juliaalv/curso_proknow.git"
import os, subprocess, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/curso_proknowc"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPOSITORIO, "/content/curso_proknowc"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "/content/curso_proknowc"], check=True)
    sys.path.insert(0, "/content/curso_proknowc/src")
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, display
from curso_proknowc import colab

pastas = colab.preparar()
print("Tudo pronto. Pasta do curso:", pastas.raiz)

In [ ]:
#@title Leitura dos arquivos { display-mode: "form" }
#@markdown **meus arquivos**: os CSVs que você colocou em `curso_proknowc/entrada/` no Google Drive.
#@markdown **exemplo do curso**: os CSVs de reserva do curso (tema curtailment de eólica e solar).
#@markdown **FONTE**: escolha uma das opções na lista (não digite o caminho de uma pasta).
FONTE = "meus arquivos" #@param ["meus arquivos", "exemplo do curso"]
from curso_proknowc.leitura import ler_pasta
tabela_bruta, arquivos = ler_pasta(colab.pasta_entrada(pastas, FONTE))
display(arquivos)
if (arquivos["situacao"] == "lido").sum() == 0:
    raise SystemExit(f"Nenhum CSV foi lido. Coloque os arquivos em {pastas.entrada / 'scopus'} e {pastas.entrada / 'ieee'} e rode esta célula de novo.")
print(f"{len(tabela_bruta)} registros lidos e padronizados.")

In [ ]:
#@title Delimitações (opcional) { display-mode: "form" }
#@markdown Deixe 0 ou em branco para não aplicar o filtro. Tipos: artigo; revisão; conferência; capítulo; outro. Idiomas: en; pt; es (registros sem idioma informado são mantidos).
ANO_INICIAL = 0 #@param {type:"integer"}
ANO_FINAL = 0 #@param {type:"integer"}
TIPOS = "" #@param {type:"string"}
IDIOMAS = "" #@param {type:"string"}
from curso_proknowc import rastreabilidade
from curso_proknowc.delimitacoes import aplicar_delimitacoes, remover_incompletos
lista = lambda campo: [t.strip() for t in campo.split(";") if t.strip()]
completos, incompletos = remover_incompletos(tabela_bruta)
anos = (ANO_INICIAL or 0, ANO_FINAL or 9999) if (ANO_INICIAL or ANO_FINAL) else None
padronizada, fora = aplicar_delimitacoes(completos, anos=anos, tipos=lista(TIPOS), idiomas=lista(IDIOMAS))
colab.registrar_bases(pastas, rastreabilidade.registros_por_base(tabela_bruta))
colab.registrar_etapa(pastas, "02", "Registros incompletos", len(tabela_bruta), incompletos)
colab.registrar_etapa(pastas, "02", "Delimitações", len(completos), fora)
print(f"Sem informação mínima: {len(incompletos)}. Fora das delimitações: {len(fora)}. Seguem: {len(padronizada)}.")

In [ ]:
#@title Relatório de qualidade e salvamento { display-mode: "form" }
BAIXAR_CSV = False #@param {type:"boolean"}
from curso_proknowc.relatorio import relatorio_qualidade
por_arquivo, avisos = relatorio_qualidade(padronizada, pd.concat([incompletos, fora]), arquivos)
display(por_arquivo)
for aviso in avisos:
    print("•", aviso)
colab.salvar(pastas, padronizada, "02_padronizado", baixar=BAIXAR_CSV)
print(f"\n{len(padronizada)} registros salvos em {pastas.resultados}.")